# 04 - Metadatos, linaje y privacidad — versión de clase

## Objetivos de aprendizaje

- Distinguir glosario, diccionario y catálogo.
- Relacionar campos de consumo con fuentes, transformaciones y reglas.
- Interpretar una huella y un documento de linaje sin atribuirles garantías falsas.
- Comparar minimización, seudonimización y denegación de acceso simulada.

No publicaremos un lote ni escribiremos archivos: trabajamos con tablas y JSON en memoria. Las cuatro filas aptas son un **subconjunto de laboratorio**, no una autorización para publicar el lote original ni para descartar sus ocho rechazos.

## Cómo trabajar

Ejecuta **una celda por vez con `Shift+Enter`**, de arriba hacia abajo. Antes de avanzar, observa la salida y responde la pregunta de la sección. Las funciones aparecen donde se necesitan, no se ejecuta todo al final.

Solo necesitas Python, pandas y un kernel Jupyter/IPython. Todos los datos y funciones se crean aquí: no se importan archivos de código, no se leen archivos de entrada y no se requieren servicios ni Internet. Las comprobaciones se hacen con Python, sin pytest.

Usamos únicamente datos ficticios y dominios `.invalid`. **No los sustituyas por datos personales reales.** La fecha de corte es fija para obtener resultados reproducibles.

Para repetir la práctica, **reinicia el kernel y ejecuta desde el paso 1**. Repetir celdas aisladas puede modificar claves, variables o historial de intentos.


## 1. Preparar el entorno

Importamos las bibliotecas y fijamos la versión del contrato y la fecha de corte. `display` permite ver tablas dentro del notebook.

**Salida esperada:** Versión de pandas, contrato 1.0.0 y corte 2026-10-06 a las 12:00 UTC.

**Para conversar:** ¿Por qué una práctica reproducible no debe depender de la fecha de hoy?


In [1]:
import hashlib
import hmac
import json
import secrets

import re
from decimal import Decimal, InvalidOperation

import pandas as pd
from IPython.display import display

FECHA_CORTE = pd.Timestamp("2026-10-06T12:00:00Z")
VERSION_CONTRATO = "1.0.0"
COLUMNAS_FUENTE = {
    "fila_fuente", "id_venta", "fecha_venta", "id_producto",
    "monto", "moneda", "email", "actualizado_en",
}

print("pandas:", pd.__version__)
print("Contrato:", VERSION_CONTRATO)
print("Fecha de corte:", FECHA_CORTE)


pandas: 3.0.2
Contrato: 1.0.0
Fecha de corte: 2026-10-06 12:00:00+00:00


## 2. Crear y observar las fuentes ficticias

Creamos ventas, maestro de productos y evidencia parcial de exactitud. La referencia es una tabla independiente para comparar importes, no una verdad conocida para todas las ventas.

**Salida esperada:** 12 ventas, 2 productos y 2 importes de referencia.

**Para conversar:** ¿Qué diferencia hay entre `fila_fuente` e `id_venta`? Busca un ID duplicado.


In [2]:
def crear_datos():
    """Retorna ventas ficticias, maestro y evidencia parcial independiente."""
    filas = [
        ("v01", "1", "2026-10-05", "101", "1500.00", " ars ", "uno@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v02", "2", "2026-10-05", "102", "200.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v03", "2", "2026-10-05", "102", "250.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v04", "3", "2026-02-29", "101", "300.00", "ARS", "tres@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v05", "4", "2026-10-05", "101", "-10.00", "ARS", "cuatro@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v06", "5", "2026-10-05", "101", "500.00", "BTC", "cinco@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v07", "6", "2026-10-05", "999", "600.00", "ARS", "seis@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v08", "7", "2026-10-05", "102", "700.00", "USD", "email-invalido", "2026-10-06T10:00:00Z"),
        ("v09", "8", "2026-09-30", "102", "800.00", "ARS", " ", "2026-10-01T10:00:00Z"),
        ("v10", "9", "2026-10-07", "101", "900.00", "ARS", None, "2026-10-06T10:00:00Z"),
        ("v11", "10", "2026-10-05", "101", "1000.00", "ARS", "diez@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v12", " ", "2026-10-05", "101", "1200.00", "ARS", None, "2026-10-06T10:00:00Z"),
    ]
    ventas = pd.DataFrame(filas, columns=[
        "fila_fuente", "id_venta", "fecha_venta", "id_producto", "monto",
        "moneda", "email", "actualizado_en",
    ])
    productos = pd.DataFrame({"id_producto": [101, 102], "producto": ["Producto A", "Producto B"]})
    referencia = pd.DataFrame({"id_venta": [1, 10], "monto_centavos_referencia": [150000, 99900]})
    return ventas, productos, referencia

crudos, productos, referencia = crear_datos()
crudos_preservados = crudos.copy(deep=True)
print("Ventas de entrada:", len(crudos))
display(crudos)
print("Maestro de productos:")
display(productos)
print("Evidencia parcial de exactitud:")
display(referencia)
assert len(crudos) == 12 and len(referencia) == 2


Ventas de entrada: 12


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,v01,1,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
1,v02,2,2026-10-05,102,200.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
2,v03,2,2026-10-05,102,250.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
3,v04,3,2026-02-29,101,300.00,ARS,tres@demo.invalid,2026-10-06T10:00:00Z
4,v05,4,2026-10-05,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06T10:00:00Z
5,v06,5,2026-10-05,101,500.00,BTC,cinco@demo.invalid,2026-10-06T10:00:00Z
6,v07,6,2026-10-05,999,600.00,ARS,seis@demo.invalid,2026-10-06T10:00:00Z
7,v08,7,2026-10-05,102,700.00,USD,email-invalido,2026-10-06T10:00:00Z
8,v09,8,2026-09-30,102,800.00,ARS,,2026-10-01T10:00:00Z
9,v10,9,2026-10-07,101,900.00,ARS,NaN,2026-10-06T10:00:00Z


Maestro de productos:


,id_producto,producto
0,101,Producto A
1,102,Producto B


Evidencia parcial de exactitud:


,id_venta,monto_centavos_referencia
0,1,150000
1,10,99900


## 3. Normalizar sin modificar el origen

Primero validamos el esquema. Luego convertimos IDs a enteros anulables, fechas a UTC, importes a centavos con `Decimal` y monedas a mayúsculas. Una fecha imposible queda como `NaT`; no inventamos una fecha válida. Normalizar no significa aprobar los datos.

**Salida esperada:** Tabla normalizada con `monto_centavos`; el origen permanece igual.

**Para conversar:** ¿Por qué no conviene calcular dinero usando números de punto flotante?


In [3]:
def esquema_valido(datos):
    return set(datos.columns) == COLUMNAS_FUENTE



def centavos(valor):
    try:
        numero = Decimal(str(valor).strip())
        if not numero.is_finite() or numero != numero.quantize(Decimal("0.01")):
            return None
        return int(numero * 100)
    except (InvalidOperation, ValueError, TypeError):
        return None



def entero_positivo(valor):
    texto = str(valor).strip()
    return int(texto) if re.fullmatch(r"[1-9][0-9]*", texto) else None



def normalizar(datos):
    if not esquema_valido(datos):
        raise ValueError("Deriva de esquema: revisar contrato antes de transformar")
    if datos["fila_fuente"].isna().any() or datos["fila_fuente"].duplicated().any():
        raise ValueError("La identificacion tecnica de origen debe ser unica y no nula")
    resultado = datos.copy(deep=True)
    for columna in ["id_venta", "id_producto"]:
        resultado[columna] = pd.array(resultado[columna].map(entero_positivo), dtype="Int64")
    resultado["monto_centavos"] = pd.array(resultado["monto"].map(centavos), dtype="Int64")
    resultado["fecha_venta"] = pd.to_datetime(resultado["fecha_venta"], format="%Y-%m-%d", errors="coerce", utc=True)
    resultado["actualizado_en"] = pd.to_datetime(resultado["actualizado_en"], format="ISO8601", errors="coerce", utc=True)
    resultado["moneda"] = resultado["moneda"].astype("string").str.strip().str.upper()
    resultado["email"] = resultado["email"].astype("string").str.strip().replace("", pd.NA)
    return resultado

datos = normalizar(crudos)
display(datos[[
    "fila_fuente", "id_venta", "fecha_venta",
    "monto_centavos", "moneda", "email",
]])
pd.testing.assert_frame_equal(crudos, crudos_preservados)
print("El origen se conserva sin cambios.")


,fila_fuente,id_venta,fecha_venta,monto_centavos,moneda,email
0,v01,1,2026-10-05 00:00:00+00:00,150000,ARS,uno@demo.invalid
1,v02,2,2026-10-05 00:00:00+00:00,20000,ARS,dos@demo.invalid
2,v03,2,2026-10-05 00:00:00+00:00,25000,ARS,dos@demo.invalid
3,v04,3,NaT,30000,ARS,tres@demo.invalid
4,v05,4,2026-10-05 00:00:00+00:00,-1000,ARS,cuatro@demo.invalid
5,v06,5,2026-10-05 00:00:00+00:00,50000,BTC,cinco@demo.invalid
6,v07,6,2026-10-05 00:00:00+00:00,60000,ARS,seis@demo.invalid
7,v08,7,2026-10-05 00:00:00+00:00,70000,USD,email-invalido
8,v09,8,2026-09-30 00:00:00+00:00,80000,ARS,<NA>
9,v10,9,2026-10-07 00:00:00+00:00,90000,ARS,<NA>


El origen se conserva sin cambios.


## 4. Evaluar calidad y separar filas

Cada regla tiene dimensión y acción. Las fallas de cuarentena rechazan la fila; las advertencias no la rechazan. Una fila puede generar varios hallazgos. En exactitud solo se evalúan las filas que tienen referencia: no evaluable no significa correcto.

La función devuelve cuatro objetos: aceptadas, rechazadas, métricas y detalle. Observa cómo las máscaras de cada regla generan el detalle y cómo se reúnen los identificadores rechazados.

**Salida esperada:** 4 filas aptas, 8 rechazadas y 17 hallazgos; 2 filas evaluables en exactitud.

**Para conversar:** ¿Puede una fila apta conservar una advertencia de email o exactitud?


In [4]:
def evaluar(datos, productos, referencia):
    """Evalua todas las filas; exactitud solo sobre evidencia disponible."""
    referencia = referencia.copy()
    if referencia["id_venta"].duplicated().any() or referencia["id_venta"].isna().any():
        raise ValueError("La referencia de exactitud tiene claves invalidas")
    if productos["id_producto"].duplicated().any() or productos["id_producto"].isna().any():
        raise ValueError("El maestro de productos tiene claves invalidas")
    verdad = datos["id_venta"].map(referencia.set_index("id_venta")["monto_centavos_referencia"])
    edad_horas = (FECHA_CORTE - datos["actualizado_en"]).dt.total_seconds() / 3600
    reglas = [
        ("DQ_ID", "completitud", "cuarentena", datos["id_venta"].notna(), None),
        ("DQ_UNICIDAD", "unicidad", "cuarentena", ~datos["id_venta"].duplicated(keep=False) & datos["id_venta"].notna(), None),
        ("DQ_FECHA", "validez", "cuarentena", datos["fecha_venta"].notna() & datos["fecha_venta"].le(FECHA_CORTE), None),
        ("DQ_MONTO", "validez", "cuarentena", datos["monto_centavos"].notna() & datos["monto_centavos"].gt(0), None),
        ("DQ_MONEDA", "integridad", "cuarentena", datos["moneda"].isin(["ARS", "USD", "EUR"]), None),
        ("DQ_FK_PRODUCTO", "integridad", "cuarentena", datos["id_producto"].isin(productos["id_producto"]), None),
        ("DQ_ACTUALIZACION", "consistencia", "cuarentena", datos["actualizado_en"].notna() & datos["actualizado_en"].le(FECHA_CORTE) & datos["actualizado_en"].ge(datos["fecha_venta"]), None),
        ("DQ_EMAIL", "validez", "advertencia", datos["email"].str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+").fillna(False), None),
        ("DQ_VIGENCIA", "vigencia", "advertencia", edad_horas.between(0, 48), None),
        ("DQ_EXACTITUD", "exactitud", "advertencia", datos["monto_centavos"].eq(verdad), verdad.notna()),
    ]
    detalle, resumen = [], []
    for regla_id, dimension, accion, cumple, elegible in reglas:
        elegible = pd.Series(True, index=datos.index) if elegible is None else elegible
        cumple = cumple.fillna(False)
        falla = elegible & ~cumple
        evaluados = int(elegible.sum())
        fallidos = int(falla.sum())
        resumen.append({
            "regla": regla_id, "dimension": dimension, "accion": accion,
            "evaluados": evaluados, "fallidos": fallidos,
            "no_evaluables": len(datos) - evaluados,
            "cumplimiento_pct": 100 * (evaluados - fallidos) / evaluados if evaluados else None,
            "owner": "Gerencia Comercial", "contrato": VERSION_CONTRATO,
        })
        for fila_fuente in datos.loc[falla, "fila_fuente"]:
            detalle.append({"fila_fuente": fila_fuente, "regla": regla_id, "accion": accion})
    detalle = pd.DataFrame(detalle, columns=["fila_fuente", "regla", "accion"])
    ids_rechazados = set(detalle.loc[detalle["accion"].eq("cuarentena"), "fila_fuente"])
    rechazadas = datos[datos["fila_fuente"].isin(ids_rechazados)].copy()
    aceptadas = datos[~datos["fila_fuente"].isin(ids_rechazados)].copy()
    assert len(datos) == len(aceptadas) + len(rechazadas)
    return aceptadas, rechazadas, pd.DataFrame(resumen), detalle

aceptadas, rechazadas, metricas, detalle = evaluar(datos, productos, referencia)
display(metricas)
display(detalle)
print("Aptas:", len(aceptadas), "| Rechazadas:", len(rechazadas))
display(aceptadas[["fila_fuente", "id_venta", "monto_centavos", "moneda"]])
assert (len(aceptadas), len(rechazadas), len(detalle)) == (4, 8, 17)
assert int(metricas.set_index("regla").loc["DQ_EXACTITUD", "evaluados"]) == 2


,regla,dimension,accion,evaluados,fallidos,no_evaluables,cumplimiento_pct,owner,contrato
0,DQ_ID,completitud,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
1,DQ_UNICIDAD,unicidad,cuarentena,12,3,0,75.000000,Gerencia Comercial,1.0.0
2,DQ_FECHA,validez,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
3,DQ_MONTO,validez,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
4,DQ_MONEDA,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
5,DQ_FK_PRODUCTO,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
6,DQ_ACTUALIZACION,consistencia,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
7,DQ_EMAIL,validez,advertencia,12,4,0,66.666667,Gerencia Comercial,1.0.0
8,DQ_VIGENCIA,vigencia,advertencia,12,1,0,91.666667,Gerencia Comercial,1.0.0
9,DQ_EXACTITUD,exactitud,advertencia,2,1,10,50.000000,Gerencia Comercial,1.0.0


,fila_fuente,regla,accion
0,v12,DQ_ID,cuarentena
1,v02,DQ_UNICIDAD,cuarentena
2,v03,DQ_UNICIDAD,cuarentena
3,v12,DQ_UNICIDAD,cuarentena
4,v04,DQ_FECHA,cuarentena
5,v10,DQ_FECHA,cuarentena
6,v05,DQ_MONTO,cuarentena
7,v06,DQ_MONEDA,cuarentena
8,v07,DQ_FK_PRODUCTO,cuarentena
9,v04,DQ_ACTUALIZACION,cuarentena


Aptas: 4 | Rechazadas: 8


,fila_fuente,id_venta,monto_centavos,moneda
0,v01,1,150000,ARS
7,v08,7,70000,USD
8,v09,8,80000,ARS
10,v11,10,100000,ARS


## 5. Definir un glosario de negocio

El glosario explica conceptos para personas de negocio. No describe todavía tipos de columnas ni permisos. `json_seguro` genera JSON legible y rechaza NaN, pero su nombre **no significa que oculte secretos o datos personales**.

**Salida esperada:** Tres términos con sus definiciones.

**Para conversar:** ¿Una venta identifica una persona? ¿Se pueden sumar importes de monedas distintas?


In [5]:
def json_seguro(objeto):
    return json.dumps(objeto, ensure_ascii=True, indent=2, allow_nan=False)

glosario = {
    "Venta": "Evento identificado por id_venta; no identifica una persona.",
    "Importe de linea": "Valor positivo expresado en la moneda original.",
    "Fila apta": "Sin fallas de cuarentena; puede conservar advertencias.",
}
print(json_seguro(glosario))
assert len(glosario) == 3


{
  "Venta": "Evento identificado por id_venta; no identifica una persona.",
  "Importe de linea": "Valor positivo expresado en la moneda original.",
  "Fila apta": "Sin fallas de cuarentena; puede conservar advertencias."
}


## 6. Construir el catálogo y el diccionario

El catálogo identifica activo, grano, finalidad, Owner, Steward y clasificación. Su lista de columnas funciona como diccionario: tipo, nulabilidad, término, origen y transformación. La retención queda **pendiente de aprobación**: no inventamos un plazo universal ni declaramos el activo listo para producción.

**Salida esperada:** Catálogo `silver.ventas` y tabla de sus cinco campos de consumo.

**Para conversar:** ¿En qué se diferencian el Owner y el Steward? ¿Qué decisión falta para producción?


In [6]:
def metadatos():
    return {
        'activo': 'silver.ventas', 'version': '1.0.0', 'grano': 'una fila por id_venta',
        'owner': 'Gerencia Comercial', 'steward': 'Steward Comercial',
        'finalidad': 'analitica de ventas', 'clasificacion': 'confidencial',
        'retencion': 'por definir y aprobar antes de produccion',
        'columnas': [
            {'nombre': 'id_venta', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Identificador de venta', 'origen': 'origen.id_venta', 'transformacion': 'entero positivo', 'reglas': ['DQ_ID', 'DQ_UNICIDAD']},
            {'nombre': 'id_producto', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Producto de la venta', 'origen': 'origen.id_producto', 'transformacion': 'lookup en productos', 'reglas': ['DQ_FK_PRODUCTO']},
            {'nombre': 'fecha_venta', 'tipo': 'DATE', 'nullable': False, 'termino': 'Fecha del evento', 'origen': 'origen.fecha_venta', 'transformacion': 'parseo ISO', 'reglas': ['DQ_FECHA']},
            {'nombre': 'monto_centavos', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Importe de linea en moneda original', 'origen': 'origen.monto', 'transformacion': 'Decimal por 100, sin conversion de moneda', 'reglas': ['DQ_MONTO']},
            {'nombre': 'moneda', 'tipo': 'TEXT', 'nullable': False, 'termino': 'Moneda original', 'origen': 'origen.moneda', 'transformacion': 'strip y upper', 'reglas': ['DQ_MONEDA']},
        ],
    }

catalogo = metadatos()
print(json_seguro({k: v for k, v in catalogo.items() if k != "columnas"}))
display(pd.DataFrame(catalogo["columnas"]))
assert catalogo["grano"] == "una fila por id_venta"
assert len(catalogo["columnas"]) == 5
assert set(pd.DataFrame(catalogo["columnas"])["nombre"]) == {
    "id_venta", "id_producto", "fecha_venta", "monto_centavos", "moneda",
}
assert json.loads(json_seguro(catalogo)) == catalogo


{
  "activo": "silver.ventas",
  "version": "1.0.0",
  "grano": "una fila por id_venta",
  "owner": "Gerencia Comercial",
  "steward": "Steward Comercial",
  "finalidad": "analitica de ventas",
  "clasificacion": "confidencial",
  "retencion": "por definir y aprobar antes de produccion"
}


,nombre,tipo,nullable,termino,origen,transformacion,reglas
0,id_venta,INTEGER,False,Identificador de venta,origen.id_venta,entero positivo,"[DQ_ID, DQ_UNICIDAD]"
1,id_producto,INTEGER,False,Producto de la venta,origen.id_producto,lookup en productos,[DQ_FK_PRODUCTO]
2,fecha_venta,DATE,False,Fecha del evento,origen.fecha_venta,parseo ISO,[DQ_FECHA]
3,monto_centavos,INTEGER,False,Importe de linea en moneda original,origen.monto,"Decimal por 100, sin conversion de moneda",[DQ_MONTO]
4,moneda,TEXT,False,Moneda original,origen.moneda,strip y upper,[DQ_MONEDA]


## 7. Calcular y comparar huellas de contenido

SHA-256 resume la representación JSON de la tabla. Comparamos una copia idéntica y otra con un importe alterado. El orden de filas y columnas participa en esta representación: no es una comparación semántica independiente del orden. La huella no demuestra autoría, exactitud ni integridad anterior a su cálculo.

**Salida esperada:** La copia idéntica tiene la misma huella y la modificada tiene otra.

**Para conversar:** ¿Por qué una huella de contenido no reemplaza un identificador de intento?


In [7]:
def huella(datos):
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()

alterados = crudos.copy(deep=True)
alterados.loc[alterados.index[0], "monto"] = "1501.00"
comparacion_huellas = pd.DataFrame([
    {"version": "original", "sha256": huella(crudos)},
    {"version": "copia identica", "sha256": huella(crudos.copy(deep=True))},
    {"version": "copia alterada", "sha256": huella(alterados)},
])
display(comparacion_huellas)
assert huella(crudos) == huella(crudos.copy(deep=True))
assert huella(crudos) != huella(alterados)
pd.testing.assert_frame_equal(crudos, crudos_preservados)


,version,sha256
0,original,99d6adeb034a0308cb980ed47833d2fcb4d29946bd79af...
1,copia identica,99d6adeb034a0308cb980ed47833d2fcb4d29946bd79af...
2,copia alterada,e21d43c4b619a5ee1f955eb9a09e2525bbf769cd7aa4b5...


## 8. Relacionar fuente, actividad, agente y destino

El linaje conecta procedencia y transformaciones por campo. Generamos un **mapeo previsto**, no evidencia de una publicación: aquí no se carga el destino. La huella corresponde a las 12 filas crudas, no solo al subconjunto apto. El modelo se inspira en W3C PROV pero no es una serialización conforme a PROV.

**Salida esperada:** Documento con fuente, huella, actividad, agente, contrato y cinco mapeos.

**Para conversar:** ¿Cómo viaja `origen.monto` hasta `monto_centavos`? ¿Dónde dice que no hay conversión monetaria?


In [ ]:
def linaje(datos):
    return {
        'entidad_fuente': 'ventas_ficticias', 'huella': huella(datos),
        'actividad': 'normalizar_validar_publicar', 'agente': 'servicio_etl_demo',
        'alcance': 'mapeo previsto; este notebook no publica el lote',
        'contrato': VERSION_CONTRATO, 'entidad_destino': 'silver.ventas',
        'campos': metadatos()['columnas'],
        'limite': 'modelo didactico inspirado en PROV, no serializacion PROV conforme',
    }

procedencia = linaje(crudos)
print(json_seguro(procedencia))
assert procedencia["huella"] == huella(crudos)
assert procedencia["contrato"] == VERSION_CONTRATO
assert len(procedencia["campos"]) == 5


{
  "entidad_fuente": "ventas_ficticias",
  "huella": "99d6adeb034a0308cb980ed47833d2fcb4d29946bd79afa2b387fcb3a23954f3",
  "actividad": "normalizar_validar_publicar",
  "agente": "servicio_etl_demo",
  "alcance": "mapeo previsto; este notebook no publica el lote",
  "contrato": "1.0.0",
  "entidad_destino": "silver.ventas",
  "campos": [
    {
      "nombre": "id_venta",
      "tipo": "INTEGER",
      "nullable": false,
      "termino": "Identificador de venta",
      "origen": "origen.id_venta",
      "transformacion": "entero positivo",
      "reglas": [
        "DQ_ID",
        "DQ_UNICIDAD"
      ]
    },
    {
      "nombre": "id_producto",
      "tipo": "INTEGER",
      "nullable": false,
      "termino": "Producto de la venta",
      "origen": "origen.id_producto",
      "transformacion": "lookup en productos",
      "reglas": [
        "DQ_FK_PRODUCTO"
      ]
    },
    {
      "nombre": "fecha_venta",
      "tipo": "DATE",
      "nullable": false,
      "termino": "Fecha del

## 9. Preparar la seudonimización con HMAC

Creamos una clave aleatoria efímera de 32 bytes y la función de tokenización. HMAC normaliza espacios y mayúsculas del texto para esta demostración. No valida si el email existe ni corrige direcciones inválidas. No es cifrado reversible ni anonimización; los tokens siguen siendo vinculables. **Nunca se imprime ni guarda la clave.**

**Salida esperada:** Mensaje de preparación; no debe aparecer el secreto.

**Para conversar:** ¿Qué ocurrirá con los tokens cuando reinicies el kernel y cambie la clave?


In [ ]:
def token_persona(valor, clave):
    if len(clave) < 32:
        raise ValueError('Usar una clave aleatoria de al menos 32 bytes para la demo')
    if pd.isna(valor) or not str(valor).strip():
        return None
    mensaje = f'demo:email:{str(valor).strip().lower()}'.encode('utf-8')
    return hmac.new(clave, mensaje, hashlib.sha256).hexdigest()

clave = secrets.token_bytes(32)
print("HMAC preparado con clave efímera de 32 bytes; no se muestra su contenido.")
assert len(clave) == 32


HMAC preparado con clave efímera de 32 bytes; no se muestra su contenido.


## 10. Observar la vista del analista

La función aplica una política simulada: el analista recibe solo columnas necesarias para analítica, sin email y sin token. La vista se crea como copia. Esto **no autentica al usuario ni implementa RBAC real**: quien controla el kernel aún puede leer los dataframes originales.

**Salida esperada:** 4 filas y 5 columnas, sin datos de contacto ni tokens.

**Para conversar:** ¿Por qué mostrar menos columnas es útil aunque no reemplace permisos reales?


In [ ]:
def vista_por_rol(datos, rol, clave):
    columnas = ['id_venta', 'id_producto', 'fecha_venta', 'monto_centavos', 'moneda']
    if rol == 'analista':
        return datos[columnas].copy()
    if rol == 'steward':
        resultado = datos[columnas].copy()
        resultado['email_token'] = datos['email'].map(lambda valor: token_persona(valor, clave))
        return resultado
    raise PermissionError('Rol sin vista autorizada en la simulacion')

vista_analista = vista_por_rol(aceptadas, "analista", clave)
display(vista_analista)
assert vista_analista.shape == (4, 5)
assert "email" not in vista_analista.columns
assert "email_token" not in vista_analista.columns
print("Vista analítica verificada: sin email ni token.")


,id_venta,id_producto,fecha_venta,monto_centavos,moneda
0,1,101,2026-10-05 00:00:00+00:00,150000,ARS
7,7,102,2026-10-05 00:00:00+00:00,70000,USD
8,8,102,2026-09-30 00:00:00+00:00,80000,ARS
10,10,101,2026-10-05 00:00:00+00:00,100000,ARS


Vista analítica verificada: sin email ni token.


## 11. Observar la vista del Steward

El Steward recibe los cinco campos anteriores más `email_token`. Una dirección ausente produce `None`, no un identificador inventado. Una dirección mal formada puede producir token porque tokenizar no valida emails. Este acceso debe tener finalidad y autorización reales fuera de la demo.

**Salida esperada:** 4 filas y 6 columnas; tokens en lugar de email.

**Para conversar:** ¿Por qué un token también debe protegerse? ¿Qué representa su valor ausente?


In [ ]:
vista_steward = vista_por_rol(aceptadas, "steward", clave)
display(vista_steward)
assert vista_steward.shape == (4, 6)
assert "email" not in vista_steward.columns
assert "email_token" in vista_steward.columns
assert vista_steward["email_token"].isna().sum() == 1
print("Vista del Steward verificada: no expone direcciones.")


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,email_token
0,1,101,2026-10-05 00:00:00+00:00,150000,ARS,acd1651ca7a6acdff53728c90a150487ba5d2c0bcb9208...
7,7,102,2026-10-05 00:00:00+00:00,70000,USD,bb37e79ccac3867d5e312669c2bc72b1f0edf09954e75d...
8,8,102,2026-09-30 00:00:00+00:00,80000,ARS,NaN
10,10,101,2026-10-05 00:00:00+00:00,100000,ARS,be2a7eb263ae05f30926835c7bf59f024d5d61848eb47a...


Vista del Steward verificada: no expone direcciones.


## 12. Comprobar una denegación esperada

Solicitamos una vista para un rol no definido. Capturamos solo `PermissionError`, lo mostramos y comprobamos que ocurrió. No ocultamos errores inesperados.

**Salida esperada:** Mensaje de acceso denegado para `visitante`, sin traceback inesperado.

**Para conversar:** ¿Por qué no conviene dar una vista por defecto a cualquier rol desconocido?


In [12]:
acceso_denegado = False
try:
    vista_por_rol(aceptadas, "visitante", clave)
except PermissionError as error:
    acceso_denegado = True
    print("Denegación esperada:", error)
assert acceso_denegado, "El rol visitante no debería obtener una vista."


Denegación esperada: Rol sin vista autorizada en la simulacion


## 14. Cerrar la práctica y reconocer sus límites

Eliminamos la referencia a la clave. `del` **no garantiza borrado seguro de memoria**. Los datos y tokens ficticios mostrados permanecen en las salidas guardadas. No hay IAM, cifrado de disco, retención aprobada ni protección frente al dueño del proceso.

**Salida esperada:** Confirmación de origen intacto y resumen final.

**Para conversar:** ¿Qué controles institucionales faltan antes de usar datos reales?


In [ ]:
pd.testing.assert_frame_equal(crudos, crudos_preservados)
del clave
print("Práctica terminada: catálogo, linaje y vistas comprobados.")
print("Se eliminó la referencia a la clave; esto no es borrado seguro de memoria.")
print("Solo se usaron datos ficticios. No se publicaron ventas ni se escribieron archivos.")


Práctica terminada: catálogo, linaje y vistas comprobados.
Se eliminó la referencia a la clave; esto no es borrado seguro de memoria.
Solo se usaron datos ficticios. No se publicaron ventas ni se escribieron archivos.


## Síntesis y ejercicios para los alumnos

- **Glosario:** significado de negocio. **Diccionario:** descripción de campos.
  **Catálogo:** activo, responsabilidades y condiciones de uso.
- **Linaje:** procedencia y transformaciones; aquí es un mapeo didáctico previsto.
- **Minimización:** menos datos visibles. **Seudonimización:** identificador vinculable.
  Ninguna de las dos reemplaza autorización, finalidad y retención.

1. Propón frecuencia de actualización, propietario de la fuente y fecha de revisión del catálogo.
   Distingue propuestas de decisiones aprobadas.
2. Dibuja el recorrido de `monto` y explica por qué no se suman monedas diferentes.
3. Diseña un procedimiento de solicitud, aprobación, revisión y revocación de acceso del Steward.
4. Explica cómo rotar y versionar una clave sin guardarla en el notebook.
5. Propón cómo relacionar linaje, ID de intento y lote realmente publicado.

**Criterio de cierre:** 4 filas en ambas vistas, cinco campos en el catálogo,
ninguna dirección visible en las vistas y visitante rechazado.
